In [10]:
!pip install -q \
langchain \
langchain-community \
langchain-huggingface \
langchain-chroma \
chromadb \
sentence-transformers \
transformers \
accelerate \
torch \
pymupdf \
pypdf \
tiktoken \
huggingface_hub \
python-dotenv \
ipywidgets \
tqdm


print(" All required libraries installed successfully!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 90.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6

In [11]:
import os
import time
import warnings
from pathlib import Path
from typing import List

from tqdm.auto import tqdm

from langchain_community.document_loaders import PyMuPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_chroma import Chroma

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    pipeline,
)

from langchain_huggingface import HuggingFacePipeline

from langchain_core.prompts import PromptTemplate

from langchain_core.output_parsers import StrOutputParser

from dotenv import load_dotenv

from IPython.display import display, Markdown

warnings.filterwarnings("ignore")

load_dotenv()


False

In [72]:
from pathlib import Path

BASE_DIR = Path.cwd()

DATA_DIR = BASE_DIR / "data"
VECTOR_DB_DIR = BASE_DIR / "chroma_db"
OUTPUT_DIR = BASE_DIR / "outputs"

CHUNK_SIZE = 500
CHUNK_OVERLAP = 100

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

TOP_K = 4

SEARCH_TYPE = "similarity"

LLM_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

MAX_NEW_TOKENS = 256
TEMPERATURE = 0.1

RANDOM_STATE = 42

DATA_DIR.mkdir(exist_ok=True)
VECTOR_DB_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

print(" PROJECT CONFIGURATION")


print(f"Base Directory      : {BASE_DIR}")
print(f"Data Folder         : {DATA_DIR}")
print(f"Vector Database     : {VECTOR_DB_DIR}")
print(f"Output Folder       : {OUTPUT_DIR}")


print(f"Chunk Size          : {CHUNK_SIZE}")
print(f"Chunk Overlap       : {CHUNK_OVERLAP}")


print(f"Embedding Model     : {EMBEDDING_MODEL}")
print(f"Retriever           : {SEARCH_TYPE}")
print(f"Top K Chunks        : {TOP_K}")


print(f"LLM Model           : {LLM_MODEL}")
print(f"Temperature         : {TEMPERATURE}")
print(f"Max Tokens          : {MAX_NEW_TOKENS}")


print(" Configuration Loaded Successfully")

 PROJECT CONFIGURATION
Base Directory      : /content
Data Folder         : /content/data
Vector Database     : /content/chroma_db
Output Folder       : /content/outputs
Chunk Size          : 500
Chunk Overlap       : 100
Embedding Model     : sentence-transformers/all-MiniLM-L6-v2
Retriever           : similarity
Top K Chunks        : 4
LLM Model           : Qwen/Qwen2.5-0.5B-Instruct
Temperature         : 0.1
Max Tokens          : 256
 Configuration Loaded Successfully


In [73]:
import os
from pathlib import Path
import sys
import shutil


print(" Scanning Data Folder")


if not DATA_DIR.exists():
    DATA_DIR.mkdir(exist_ok=True)
    print(f"Created data directory: {DATA_DIR}")

pdf_files = sorted(DATA_DIR.glob("*.pdf"))

is_colab = 'google.colab' in sys.modules

if len(pdf_files) == 0 and is_colab:
    print("\nNo PDF files found in data directory. Opening file upload dialog...")
    from google.colab import files

    uploaded = files.upload()

    if uploaded:
        print("\nMoving uploaded files to the data directory...")
        for filename in uploaded.keys():
            shutil.move(filename, DATA_DIR / filename)
        print(" Files moved successfully!")
        pdf_files = sorted(DATA_DIR.glob("*.pdf"))
    else:
        print("No files were uploaded.")


if len(pdf_files) == 0:
    raise FileNotFoundError(
        f"""
 No PDF files found.

Please place one or more PDF files inside:

{DATA_DIR}

If you are in Google Colab, an upload dialog should have appeared.
"""
    )

print(f" Found {len(pdf_files)} PDF file(s)\n")


print(f"{'No.':<5}{'Filename':<40}{'Size (MB)':>12}")


total_size = 0

for idx, pdf in enumerate(pdf_files, start=1):

    size_mb = pdf.stat().st_size / (1024 * 1024)
    total_size += size_mb

    print(f"{idx:<5}{pdf.name:<40}{size_mb:>10.2f}")



print(f" Total PDFs     : {len(pdf_files)}")
print(f" Total Size     : {total_size:.2f} MB")
print(f" Folder         : {DATA_DIR}")


print(" PDF Validation Completed Successfully")

 Scanning Data Folder
 Found 2 PDF file(s)

No.  Filename                                   Size (MB)
1    Kunchakuri_Sathwik_Resume (1).pdf             0.29
2    Mandala Chandrakanth Reddy.pdf                0.21
 Total PDFs     : 2
 Total Size     : 0.50 MB
 Folder         : /content/data
 PDF Validation Completed Successfully


In [74]:
from google.colab import files
import shutil
from pathlib import Path

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

uploaded = files.upload()

for filename in uploaded:
    destination = DATA_DIR / filename
    shutil.move(filename, destination)
    print(f"Added: {destination.name}")

print("\nCurrent PDFs:")
for pdf in sorted(DATA_DIR.glob("*.pdf")):
    print("-", pdf.name)


Current PDFs:
- Kunchakuri_Sathwik_Resume (1).pdf
- Mandala Chandrakanth Reddy.pdf


In [75]:

print(" Loading PDF Documents")


import time

start_time = time.time()

documents = []
total_pages = 0

for pdf_path in pdf_files:

    print(f"\n Loading : {pdf_path.name}")

    try:
        loader = PyMuPDFLoader(str(pdf_path))
        pages = loader.load()

        for page in pages:

            page.metadata["document_name"] = pdf_path.name
            page.metadata["file_path"] = str(pdf_path)

        documents.extend(pages)

        total_pages += len(pages)

        print(f"   Pages Loaded : {len(pages)}")

    except Exception as e:

        print(f"   Failed to load {pdf_path.name}")
        print(f"   Error : {e}")

loading_time = time.time() - start_time


print(" DOCUMENT LOADING SUMMARY")


print(f"Total PDFs Loaded     : {len(pdf_files)}")
print(f"Total Pages Loaded    : {total_pages}")
print(f"Document Objects      : {len(documents)}")
print(f"Loading Time          : {loading_time:.2f} seconds")


if len(documents) > 0:

    sample = documents[0]

    print("\n SAMPLE DOCUMENT METADATA")


    print(f"Document Name : {sample.metadata.get('document_name')}")
    print(f"Page Number   : {sample.metadata.get('page', 'N/A') + 1}")
    print(f"Source        : {sample.metadata.get('source')}")

    print("\n Preview (First 500 Characters)")


    print(sample.page_content[:500])

    print("\n...")

print("\n Document Loading Completed Successfully!")

 Loading PDF Documents

 Loading : Kunchakuri_Sathwik_Resume (1).pdf
   Pages Loaded : 2

 Loading : Mandala Chandrakanth Reddy.pdf
   Pages Loaded : 2
 DOCUMENT LOADING SUMMARY
Total PDFs Loaded     : 2
Total Pages Loaded    : 4
Document Objects      : 4
Loading Time          : 0.10 seconds

 SAMPLE DOCUMENT METADATA
Document Name : Kunchakuri_Sathwik_Resume (1).pdf
Page Number   : 1
Source        : /content/data/Kunchakuri_Sathwik_Resume (1).pdf

 Preview (First 500 Characters)
Kunchakuri Sathwik
Hyderabad, India | kunchakurisathwik414@gmail.com | +91-9014596793 | LinkedIn | GitHub
P R O F E S S I O N A L S U M M A R Y
I am a final-year B.Tech student specialising in Artificial Intelligence and Machine Learning, with hands-on experience building
real-world AI applications. My core strength lies in designing end-to-end solutions using Large Language Models and
Retrieval-Augmented Generation — from raw prototypes to production-ready systems. I enjoy tackling complex prob

...

 Documen

In [77]:

print(" Splitting Documents into Chunks")

start_time = time.time()

text_splitter = RecursiveCharacterTextSplitter(

    chunk_size = 450,
    chunk_overlap = 75,

    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ],

    length_function=len
)

chunks = text_splitter.split_documents(documents)

chunking_time = time.time() - start_time

print(f"\n Chunking Completed Successfully!")


print(" CHUNKING SUMMARY")


print(f"Original Documents : {len(documents)}")
print(f"Generated Chunks   : {len(chunks)}")
print(f"Chunk Size         : {text_splitter._chunk_size}")
print(f"Chunk Overlap      : {text_splitter._chunk_overlap}")
print(f"Processing Time    : {chunking_time:.2f} sec")



sample_chunk = chunks[0]

print("\n FIRST CHUNK PREVIEW")


print(sample_chunk.page_content[:700])

print("\n...")

print("\n Metadata")

for key, value in sample_chunk.metadata.items():
    print(f"{key:20}: {value}")

print("\n Ready for Embedding Generation")

 Splitting Documents into Chunks

 Chunking Completed Successfully!
 CHUNKING SUMMARY
Original Documents : 4
Generated Chunks   : 15
Chunk Size         : 450
Chunk Overlap      : 75
Processing Time    : 0.00 sec

 FIRST CHUNK PREVIEW
Kunchakuri Sathwik
Hyderabad, India | kunchakurisathwik414@gmail.com | +91-9014596793 | LinkedIn | GitHub
P R O F E S S I O N A L S U M M A R Y
I am a final-year B.Tech student specialising in Artificial Intelligence and Machine Learning, with hands-on experience building
real-world AI applications. My core strength lies in designing end-to-end solutions using Large Language Models and

...

 Metadata
producer            : Skia/PDF m128
creator             : 
creationdate        : 
source              : /content/data/Kunchakuri_Sathwik_Resume (1).pdf
file_path           : /content/data/Kunchakuri_Sathwik_Resume (1).pdf
total_pages         : 2
format              : PDF 1.4
title               : 
author              : 
subject             : 
keywords        

In [79]:
from collections import Counter

counter = Counter()

for chunk in chunks:
    counter[chunk.metadata["document_name"]] += 1

print(counter)

Counter({'Kunchakuri_Sathwik_Resume (1).pdf': 10, 'Mandala Chandrakanth Reddy.pdf': 5})


In [81]:

print(" Loading Embedding Model")



try:

    embeddings = HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL,
        model_kwargs={"device": "cpu"},
        encode_kwargs={
            "normalize_embeddings": True
        }
    )


    print(" Embedding Model Loaded Successfully!\n")

    print(" EMBEDDING MODEL INFORMATION")

    print(f"Model Name       : {EMBEDDING_MODEL}")


except Exception as e:

    print(" Failed to load embedding model")
    print(e)

 Loading Embedding Model


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

 Embedding Model Loaded Successfully!

 EMBEDDING MODEL INFORMATION
Model Name       : sentence-transformers/all-MiniLM-L6-v2


In [28]:
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 51.3 MB/s eta 0:00:00


In [82]:

print("Creating FAISS Vector Database")

import time
import os
import shutil

from langchain_community.vectorstores import FAISS

start_time = time.time()



vector_db = FAISS.from_documents(

    documents=chunks,

    embedding=embeddings

)

creation_time = time.time() - start_time


faiss_path = "faiss_index"

if os.path.exists(faiss_path):
    shutil.rmtree(faiss_path)

vector_db.save_local(faiss_path)


print(" FAISS VECTOR DATABASE CREATED SUCCESSFULLY")

print(f"Stored Chunks      : {len(chunks)}")
print(f"Vector Database    : FAISS")
print(f"Index Path         : {faiss_path}")
print(f"Embedding Model    : {EMBEDDING_MODEL}")


print("\n Verifying Vector Database...")

test_docs = vector_db.similarity_search(
    "test",
    k=2
)

print(f"Retrieved Documents : {len(test_docs)}")

print("\n FAISS Database Ready for Retrieval")


Creating FAISS Vector Database
 FAISS VECTOR DATABASE CREATED SUCCESSFULLY
Stored Chunks      : 15
Vector Database    : FAISS
Index Path         : faiss_index
Embedding Model    : sentence-transformers/all-MiniLM-L6-v2

 Verifying Vector Database...
Retrieved Documents : 2

 FAISS Database Ready for Retrieval


In [83]:

print("Creating Production Hybrid Retriever")

from langchain_community.retrievers import BM25Retriever


similarity_retriever = vector_db.as_retriever(

    search_type="similarity",

    search_kwargs={
        "k":3
    }

)



mmr_retriever = vector_db.as_retriever(

    search_type="mmr",

    search_kwargs={
        "k":3,
        "fetch_k":10,
        "lambda_mult":0.5
    }

)



bm25_retriever = BM25Retriever.from_documents(chunks)

bm25_retriever.k = 3


class ProductionHybridRetriever:

    def __init__(
        self,
        similarity_retriever,
        mmr_retriever,
        bm25_retriever,
        top_k=4
    ):

        self.similarity = similarity_retriever
        self.mmr = mmr_retriever
        self.bm25 = bm25_retriever
        self.top_k = top_k

    def invoke(self, query):

        similarity_docs = self.similarity.invoke(query)

        mmr_docs = self.mmr.invoke(query)

        bm25_docs = self.bm25.invoke(query)

        merged_docs = (
            similarity_docs +
            mmr_docs +
            bm25_docs
        )

        unique_docs = []

        seen = set()

        for doc in merged_docs:

            key = (

                doc.metadata.get("document_name"),

                doc.metadata.get("page", 0),

                doc.page_content[:120]

            )

            if key not in seen:

                seen.add(key)

                unique_docs.append(doc)

        return unique_docs[:self.top_k]



retriever = ProductionHybridRetriever(

    similarity_retriever,

    mmr_retriever,

    bm25_retriever,

    top_k=4

)



Creating Production Hybrid Retriever


In [84]:

print(" Testing the Retriever")

query = "What are Sathwik's technical skills?"

print(f"\n Query:\n{query}")

start_time = time.time()

retrieved_docs = retriever.invoke(query);

retrieval_time = time.time() - start_time


print(f" Retrieved {len(retrieved_docs)} Relevant Chunks")
print(f" Retrieval Time : {retrieval_time:.4f} seconds")

for idx, doc in enumerate(retrieved_docs, start=1):

    metadata = doc.metadata

    page_number = metadata.get("page", 0) + 1

    document_name = metadata.get("document_name", "Unknown")

    print(f"\n Chunk {idx}")

    print(f"Document : {document_name}")
    print(f"Page     : {page_number}")

    preview = doc.page_content.replace("\n", " ")

    print("\nContent Preview:\n")

    print(preview[:400])


print("\n Retriever Test Completed Successfully!")

 Testing the Retriever

 Query:
What are Sathwik's technical skills?
 Retrieved 4 Relevant Chunks
 Retrieval Time : 0.0464 seconds

 Chunk 1
Document : Kunchakuri_Sathwik_Resume (1).pdf
Page     : 1

Content Preview:

Kunchakuri Sathwik Hyderabad, India | kunchakurisathwik414@gmail.com | +91-9014596793 | LinkedIn | GitHub P R O F E S S I O N A L S U M M A R Y I am a final-year B.Tech student specialising in Artificial Intelligence and Machine Learning, with hands-on experience building real-world AI applications. My core strength lies in designing end-to-end solutions using Large Language Models and

 Chunk 2
Document : Kunchakuri_Sathwik_Resume (1).pdf
Page     : 1

Content Preview:

Retrieval-Augmented Generation — from raw prototypes to production-ready systems. I enjoy tackling complex problems, having solved over 200 coding challenges that sharpen my algorithmic thinking. I am actively seeking AI/ML Engineer roles where I can contribute meaningfully from day one. T E C H N I C A L

In [86]:


import torch
import time

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline,
)

from langchain_huggingface import HuggingFacePipeline

start_time = time.time()

try:

    print("Loading tokenizer...")

    tokenizer = AutoTokenizer.from_pretrained(
        LLM_MODEL,
        trust_remote_code=True
    )

    print("Loading model...")

    model = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL,
        torch_dtype="auto",
        device_map="auto",
        trust_remote_code=True
    )

    print("Creating pipeline...")

    text_pipeline = pipeline(

        task="text-generation",

        model=model,

        tokenizer=tokenizer,

        max_new_tokens=256,

        do_sample=False,

        temperature=0.1,

        repetition_penalty=1.1,

        return_full_text=False

    )

    llm = HuggingFacePipeline(
        pipeline=text_pipeline
    )

    loading_time = time.time() - start_time

    print(" Language Model Loaded Successfully!")

    print(f"Model            : {LLM_MODEL}")
    print(f"Temperature      : 0.1")
    print(f"Max New Tokens   : 256")




except Exception as e:

    print("\n Failed to load model")

    print(type(e).__name__)

    print(e)

Loading tokenizer...
Loading model...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Creating pipeline...
 Language Model Loaded Successfully!
Model            : Qwen/Qwen2.5-0.5B-Instruct
Temperature      : 0.1
Max New Tokens   : 256


In [87]:

print(" Creating RAG Prompt")

PROMPT_TEMPLATE = """
You are an expert AI assistant for Document Question Answering.

Your task is to answer the user's question ONLY using the provided context.

Instructions:

1. Use only the information present in the context.
2. Do not add outside knowledge.
3. If the answer is not available in the context, reply exactly:

"I couldn't find this information in the provided documents."

4. Answer in clear English.
5. Use bullet points whenever appropriate.
6. If possible, mention the source document naturally.

==================================================
CONTEXT
==================================================

{context}

==================================================
QUESTION
==================================================

{question}

==================================================
ANSWER
==================================================
"""

print(" Prompt Template Created Successfully!")

print("\nPrompt Length :", len(PROMPT_TEMPLATE), "characters")

print("\nPrompt Preview")

print(PROMPT_TEMPLATE[:500])

print("\n...")

 Creating RAG Prompt
 Prompt Template Created Successfully!

Prompt Length : 854 characters

Prompt Preview

You are an expert AI assistant for Document Question Answering.

Your task is to answer the user's question ONLY using the provided context.

Instructions:

1. Use only the information present in the context.
2. Do not add outside knowledge.
3. If the answer is not available in the context, reply exactly:

"I couldn't find this information in the provided documents."

4. Answer in clear English.
5. Use bullet points whenever appropriate.
6. If possible, mention the source document naturally.

=

...


In [89]:
# ============================================================
# RAG DOCUMENT QUESTION ANSWERING SYSTEM
# Cell 15 : Interactive Question Answering (Production Version)
# ============================================================

import time

print("RAG Document Question Answering System Ready!")

while True:

    question = input(" Ask a question: ").strip()

    if question.lower() == "exit":
        print("\n Thank you for using the RAG System!")
        break

    if len(question) == 0:
        print("Please enter a valid question.")
        continue


    results = vector_db.similarity_search_with_score(
        question,
        k=15
    )


    print("Retrieved Chunks")

    retrieved_docs = []

    seen_chunks = set()

    for doc, score in results:

        print(
            f"{score:.4f} | "
            f"{doc.metadata.get('document_name')} | "
            f"Page {doc.metadata.get('page',0)+1}"
        )

        key = (
            doc.metadata.get("document_name"),
            doc.metadata.get("page"),
            doc.page_content[:200]
        )

        if key not in seen_chunks:

            seen_chunks.add(key)

            retrieved_docs.append(doc)

    retrieved_docs = retrieved_docs[:6]


    context_parts = []

    sources = []

    for doc in retrieved_docs:

        context_parts.append(doc.page_content)

        sources.append(
            (
                doc.metadata.get("document_name"),
                doc.metadata.get("page",0)+1
            )
        )

    context = "\n\n".join(context_parts)

    final_prompt = PROMPT_TEMPLATE.format(
        context=context,
        question=question
    )



    inputs = tokenizer(
        final_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    )

    inputs = {
        k: v.to(model.device)
        for k,v in inputs.items()
    }

    input_length = inputs["input_ids"].shape[1]



    generation_start = time.time()

    outputs = model.generate(

        **inputs,

        max_new_tokens=256,

        temperature=0.1,

        do_sample=False,

        repetition_penalty=1.15,

        pad_token_id=tokenizer.eos_token_id

    )

    generation_time = time.time() - generation_start


    answer = tokenizer.decode(

        outputs[0][input_length:],

        skip_special_tokens=True

    ).strip()

    total_time = time.time() - overall_start


    print("ANSWER")

    print(answer)
    print("SOURCES")

    unique_sources = []

    seen = set()

    for src in sources:

        if src not in seen:

            seen.add(src)

            unique_sources.append(src)

    for i,(doc,page) in enumerate(unique_sources,1):

        print(f"{i}. {doc} (Page {page})")



    print(f"Retrieved Candidates : {len(results)}")
    print(f"Unique Chunks Used   : {len(retrieved_docs)}")

RAG Document Question Answering System Ready!
 Ask a question: skills
Retrieved Chunks
1.2079 | Kunchakuri_Sathwik_Resume (1).pdf | Page 1
1.2493 | Mandala Chandrakanth Reddy.pdf | Page 1
1.3229 | Kunchakuri_Sathwik_Resume (1).pdf | Page 1
1.3356 | Mandala Chandrakanth Reddy.pdf | Page 1
1.3749 | Kunchakuri_Sathwik_Resume (1).pdf | Page 1
1.4207 | Kunchakuri_Sathwik_Resume (1).pdf | Page 1
1.4691 | Kunchakuri_Sathwik_Resume (1).pdf | Page 2
1.4840 | Kunchakuri_Sathwik_Resume (1).pdf | Page 2
1.5007 | Kunchakuri_Sathwik_Resume (1).pdf | Page 1
1.5801 | Mandala Chandrakanth Reddy.pdf | Page 1
1.6210 | Mandala Chandrakanth Reddy.pdf | Page 1
1.6561 | Mandala Chandrakanth Reddy.pdf | Page 1
1.6956 | Kunchakuri_Sathwik_Resume (1).pdf | Page 2
1.7036 | Kunchakuri_Sathwik_Resume (1).pdf | Page 1
1.7181 | Kunchakuri_Sathwik_Resume (1).pdf | Page 1
ANSWER
Skills include programming languages such as Python, Java, and SQL; artificial intelligence and machine learning skills; retrieval augmented 

In [90]:
import time
import pandas as pd

evaluation_questions = [
    {
        "question": "What are Sathwik's technical skills?",
        "keywords": ["Python", "SQL", "Java", "AI", "Machine Learning"]
    },
    {
        "question": "What internship experience does Sathwik have?",
        "keywords": ["Intern", "CodexIntern", "Python"]
    }
]

results = []

print("RAG SYSTEM EVALUATION")


for item in evaluation_questions:

    question = item["question"]

    start = time.time()

    retrieved_docs = vector_db.similarity_search(question, k=4)

    retrieval_time = time.time() - start

    context = " ".join([doc.page_content for doc in retrieved_docs])

    matched = sum(
        keyword.lower() in context.lower()
        for keyword in item["keywords"]
    )

    accuracy = matched / len(item["keywords"])

    results.append({
        "Question": question,
        "Retrieved Chunks": len(retrieved_docs),
        "Keyword Accuracy": round(accuracy, 2),
    })

df = pd.DataFrame(results)

print(df)


print(f"Average Keyword Accuracy : {df['Keyword Accuracy'].mean():.2f}")
print(f"Average Retrieved Chunks : {df['Retrieved Chunks'].mean():.2f}")


RAG SYSTEM EVALUATION
                                        Question  Retrieved Chunks  \
0           What are Sathwik's technical skills?                 4   
1  What internship experience does Sathwik have?                 4   

   Keyword Accuracy  
0               1.0  
1               1.0  
Average Keyword Accuracy : 1.00
Average Retrieved Chunks : 4.00
